# 02. 設定生成 — Visibility の理解

**前提ファイル**: `network.onnx`（`01_onnx.ipynb` または `simple_demo.ipynb` を実行済み）

In [6]:
try:
    import google.colab, subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "ezkl", "onnx", "torch", "torchvision"])
except ImportError:
    pass

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
import ezkl, json, os

model_path    = '/content/drive/MyDrive/ezkl/network.onnx'
settings_path = '/content/drive/MyDrive/ezkl/input.json'

assert os.path.exists(model_path), "network.onnx がありません。01_onnx.ipynb を先に実行してください。"

## Visibility とは

ZK 証明では「何を証明者だけが知っていて、何を検証者も知るか」を宣言する必要があります。

| visibility | 意味 | 誰が見える |
|---|---|---|
| `public` | 公開値 | 証明者・検証者の両方 |
| `private` | 秘密値 | 証明者のみ |
| `fixed` | 回路に焼き込み（変更不可） | 回路生成時に決定 |

設定する項目は 3 つ：
- `input_visibility` — 入力データ（画像など）
- `output_visibility` — 推論結果（クラス分類など）
- `param_visibility` — モデルの重み（weight, bias）

## よくある visibility の組み合わせ

| 用途 | input | output | param |
|---|---|---|---|
| 基本デモ（このノートブック） | public | public | fixed |
| 入力を隠したい（プライバシー保護） | private | public | fixed |
| モデルを隠したい（独自モデルの秘匿） | public | public | private |
| 完全秘匿 | private | public | private |

`param = fixed` は「このモデルの重みで計算した」ことを証明するためのコミットメントです。RareSkills の「モデル重みへのコミットメント」に対応します。

In [8]:
py_run_args = ezkl.PyRunArgs()
py_run_args.input_visibility  = "public"
py_run_args.output_visibility = "public"
py_run_args.param_visibility  = "fixed"

# ここで設計図を生成！
res = ezkl.gen_settings(model_path, settings_path, py_run_args=py_run_args)
print(f"gen_settings: {res}")

gen_settings: True


ONNX形式のAIモデル（network.onnx）をZKの回路に変換するとき、いきなり変換することはできません。
なぜなら、「どのデータを秘密にして、どのデータを公開するか」や「計算の精度をどうするか」といったルールがまだ決まっていないからです。

そこで、この ezkl.gen_settings を使って以下の情報を1つのファイルにまとめます。

- Visibility（公開範囲）の指定  
画面で設定している input_visibility、output_visibility、param_visibility の設定（public や fixed など）を記録します。

- モデル構造の解析  
network.onnx を読み込んで、「どんな計算（掛け算や足し算）がどれくらい含まれているか」を自動で調べます。

- 回路の規模の見積もり  
ZK証明を作るために必要な「暗号的なスペース（専門用語で多項式の次数など）」がどれくらい必要かを計算し、設定に書き込みます。

## settings.json の中身を確認する

In [9]:
with open(settings_path) as f:
    s = json.load(f)

print("=== 主要な設定項目 ===")
print(f"input_visibility  : {s['run_args']['input_visibility']}")
print(f"output_visibility : {s['run_args']['output_visibility']}")
print(f"param_visibility  : {s['run_args']['param_visibility']}")
print(f"\n=== 全キー ===")
for k in s.keys():
    print(f"  {k}")

=== 主要な設定項目 ===
input_visibility  : Public
output_visibility : Public
param_visibility  : Fixed

=== 全キー ===
  run_args
  num_rows
  total_assignments
  total_const_size
  total_dynamic_col_size
  max_dynamic_input_len
  num_dynamic_lookups
  num_shuffles
  total_shuffle_col_size
  einsum_params
  model_instance_shapes
  model_output_scales
  model_input_scales
  module_sizes
  required_lookups
  required_range_checks
  check_mode
  version
  num_blinding_factors
  timestamp
  input_types
  output_types


In [10]:
# run_args の全項目を確認
print("=== run_args の全項目 ===")
for k, v in s['run_args'].items():
    print(f"  {k}: {v}")

=== run_args の全項目 ===
  input_scale: 7
  param_scale: 7
  rebase_scale: None
  scale_rebase_multiplier: 1
  lookup_range: [-32768, 32768]
  logrows: 17
  num_inner_cols: 2
  variables: [['batch_size', 1]]
  input_visibility: Public
  output_visibility: Public
  param_visibility: Fixed
  rebase_frac_zero_constants: False
  check_mode: UNSAFE
  decomp_base: 16384
  decomp_legs: 2
  bounded_log_lookup: False
  ignore_range_check_inputs_outputs: False
  epsilon: 2.220446049250313e-16
  disable_freivalds: False


## `param_visibility = fixed` vs `private` の違い

```
fixed  → 重みを回路に直接埋め込む
         証明するたびに「この重みで計算した」が保証される
         重みは公開されるが、すり替えは不可能

private → 重みを証明者だけが知る秘密として扱う
          「何らかの重みで正しく計算した」ことだけを証明
          モデルの中身を公開せずに推論を証明できる
```

---
次: `03_calibrate.ipynb` で浮動小数点を ZK 向けの整数に変換します。